### Imports

In [26]:
import chipwhisperer as cw
import matplotlib.pyplot as plt
import numpy as np
import time
import struct
import random

from scipy.signal import find_peaks

In [2]:
# project_name = "protected1mil"
# num_traces = 1000000
# scmd_value = 1 #0 for unprotected and 1 for protected

In [91]:
project_name = "full_unprotected_encoded_simple_nops_more_dec24_nops_for_neuron"
num_traces = 2000
scmd_value = 0 #0 for unprotected and 1 for protected

In [28]:
min_in_val = -2
max_in_val = 2
decimate_value = 1
# decimate value read about for chipwhisperer

filename = project_name + "-trace.txt"

### Function Definitions

In [29]:
def random_float(min_val, max_val):
    # Generate a random float between min_val and max_val
    rand_float = random.uniform(min_val, max_val)
    # Round to 2 decimal places
    return round(rand_float, 2)

In [30]:
def float_to_bytearray_32bit_little_edian(f):
    # Pack the float as a 32-bit (4-byte) IEEE 754 floating point number
    packed = struct.pack('f', f)
    # Convert to bytearray
    return bytearray(packed)

In [31]:
def scope_setup(samples=24431, decimate=2):
    # arm the scope
    scope.arm()
    
    # Set the maximum number of points in a trace
    scope.adc.fifo_fill_mode = "normal"
    scope.adc.samples = samples
    scope.adc.decimate = decimate

In [32]:
def capture_trace(cmd_data, cmd='p', scmd=scmd_value, prints=True):
    scope.arm()
    # flush the UART buffer
    target.flush()
    
    target.send_cmd(cmd, scmd, cmd_data)
    ret = scope.capture()
    trace = scope.get_last_trace()
    
    returned_data = target.read_cmd('r')
    ack = target.read_cmd('e')
    if prints:
        print(f'r\t- target.read_cmd("r"):\t{returned_data}')
        print(f'ack\t- target.read_cmd("e"):\t{ack}')
    return trace
    

### Target Setup

In [93]:
#Scope setup
scope = cw.scope() 
scope.default_setup()

target = cw.target(scope, cw.targets.SimpleSerial2) #cw.targets.SimpleSerial can be omitted
#MY CHANGES - changed target to SimpleSerial2 - to be able to send_cmd

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 1109861                   to 11594953                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 29538471                  to 29538459                 
scope.clock.adc_rate                     changed from 29538471.0                to 29538459.0               
scope.clock.clkgen_

In [34]:
def disconnect_target():
    """Properly disconnect the target"""
    try:
        # Send disconnect command
        target.send_cmd('d', 0, b'')
        
        # Read final acknowledgement
        returned_data = target.read_cmd('r', timeout=100)
        print(f"Disconnect ACK: {returned_data}")
        
        # Short delay to ensure transmission
        time.sleep(0.1)
        
    except Exception as e:
        print(f"Disconnect warning: {e}")
    
    finally:
        # Always disconnect scope and target
        target.dis()
        scope.dis()
        print("Target and scope disconnected")

In [99]:
disconnect_target()

(ChipWhisperer Target WARNING|File SimpleSerial2.py:518) Unexpected start to command 101


Disconnect ACK: CWbytearray(b'00 65 01 01 a6 00')
Target and scope disconnected


In [94]:
scope_setup(samples=24430, decimate=30)

In [ ]:
%%bash
cd network/
make PLATFORM='CWLITEARM' CRYPTO_TARGET=NONE

In [95]:
cw.program_target(scope, cw.programmers.STM32FProgrammer, "./encoding_with_lu/encoding_with_lut_compiled/simpleserial-target-CWLITEARM.hex")

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 6335 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 6335 bytes


### Initialize the project

The Chipwhisperer `Project` class can be used to keep a collection of traces. 

In [96]:
proj = cw.create_project(project_name)
proj.traces.init(num_traces)

AttributeError: 'Traces' object has no attribute 'init'

### Trace collection

In [ ]:
trace_waves_arr = []
inputs_arr = []

start = time.time()
completed_counter = 0

# 50 dummy executions
# 50 dummy executions
# ALLOWED_INPUTS = [-7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7]
ALLOWED_INPUTS = [
    0b110100010000001,
    0b010100010000010,
    0b100000000000011,
    0b100100010000100,
    0b010000000000101,
    0b110000000000110,
    0b000100010000111,
    0b000100010001000,
    0b110000000001001,
    0b010000000001010,
    0b100100010001011,
    0b100000000001100,
    0b010100010001101,
    0b110100010001110
]

warm_quant = [random.choice(ALLOWED_INPUTS) for _ in range(10)]    
warm_bytes = bytearray()
for x in warm_quant:
    warm_bytes += x.to_bytes(2, byteorder='little', signed=False)

for i in range(50):
    trace_wave = capture_trace(warm_bytes, scmd=scmd_value)
# test_data = bytearray([1,2,3,4,5,6,7,8,9,10])
# resp = capture_trace(test_data, scmd=0x00)   # assuming 't' is the command
# print(resp)  # should see the same bytes returned

print("warm up done")

# # ===== Quantized Input Parameters =====
# QMIN_IN = -7
# QMAX_IN = 7

# Real executions
start = time.time()
completed_counter = 0

trace_waves_arr = []    # will hold trace waveforms
inputs_arr = []         # will hold lists of 10 int8 values

for i in range(num_traces):
    # Generate 10 random int8 values in the allowed range
# Allowed input values (same as allowed weights – non‑zero symmetric int4)
    # ALLOWED_INPUTS = [-7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7]
    quant_inputs = [random.choice(ALLOWED_INPUTS) for _ in range(10)]    
    inputs_arr.append(quant_inputs)
    
    # Convert to signed bytes (int8_t)
    cmd_data = bytearray()
    for x in quant_inputs:
        cmd_data += x.to_bytes(2, byteorder='little', signed=False)

    trace_wave = capture_trace(cmd_data=cmd_data, scmd=scmd_value, prints=False)
    trace_waves_arr.append(np.array(trace_wave))
    
    completed_counter += 1
    if completed_counter % 100 == 0:
        print(f"captured {completed_counter} traces in {time.time() - start:.2f} seconds")

end = time.time()
print(f'Capturing finished in {end - start:.2f} seconds!')

trace_waves_arr = np.array(trace_waves_arr)
inputs_arr = np.array(inputs_arr)   # shape (num_traces, 10), dtype=int

r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 5f ff ff ff 38 00')
ack	- target.r

In [98]:
save_files(project_name, trace_waves_arr, "inputs.txt", inputs_arr)

In [18]:
# proj.save()
proj.close()

### Plot trace

In [19]:
proj = cw.open_project(project_name)

ERROR:root:Array can't be memory-mapped: Python objects in dtype.


In [37]:
print(len(proj.traces))

0


In [46]:
trace_waves_arr = []
for trace in proj.traces:
    trace_waves_arr.append(trace.wave)

In [ ]:
import holoviews as hv
hv.extension('bokeh')
list_of_curves = [
    hv.Curve(trace_waves_arr[0], label='one trace'), 
]
hv.Overlay(list_of_curves).opts(
    height=600, 
    width=800
)

In [48]:
def disconnect_DUT():
    scope.dis()
    target.dis()
    return
disconnect_DUT()

In [ ]:
# ## save one trace to file
# f = open(filename, "w")
# i = 0
# f.write("x y\n")
# for point in trace_waves_arr[0]:
#     f.write(str(i)+" "+str(point))
#     f.write("\n")
#     i = i+1
# f.close()

### Save traces as txt files

In [ ]:
proj = cw.open_project(project_name)

In [47]:
trace_waves_arr = []
inputs_arr = []
for trace in proj.traces:
    trace_waves_arr.append(trace.wave)
    inputs_arr.append(trace.textin)

trace_waves_arr = np.array(trace_waves_arr)
print(len(trace_waves_arr))
print(len(inputs_arr))

0
0


In [39]:
import os

def save_files(folder, array, input_file, input_array):
    isExist = os.path.exists(folder)
    if not isExist:
        os.makedirs(folder)
    no_of_traces = len(input_array)
    for n in range(no_of_traces):
        with open(folder + "/trace_"+str(n)+".txt","w+") as file:
            for record in array[n]:
                file.write(str(record)+"\n")
        file.close()
    
    with open(folder + "/" + input_file,"w+") as file:
        for i in range(no_of_traces):
            line = " ".join([str(x) for x in input_array[i]])
            file.write(line + "\n")
    return

In [ ]:
print(len(trace_waves_arr[0]))

In [ ]:
save_files(project_name, trace_waves_arr, "inputs.txt", inputs_arr)

### Zip files

In [ ]:
import shutil
shutil.make_archive(project_name, 'zip', project_name)
# shutil.make_archive(output_filename_dont_add_.zip, 'zip', directory_to_download)

### Delete files

In [ ]:
import shutil

shutil.rmtree('5')

### Unzip file

In [ ]:
# import zipfile as zf
# files = zf.ZipFile("version_02.zip", 'r')
# files.extractall('network')
# files.close()